In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
final_df=pd.read_csv('D:\CA_content\Python\Final_Project\Final_Cleaned_Dataset_with_sorting.csv')

In [4]:
# Sirf wo rows rakhein jinki price 0 se zyada hai
final_df = final_df[final_df['price'] > 0]

# Verify karein
print(f"Rows after removing zero prices: {final_df.shape[0]}")

Rows after removing zero prices: 6708805


In [ ]:
uv pip install xgboost pandas numpy scikit-learn matplotlib seaborn

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("⚙️ Initiating GRU Data Transformation Pipeline...")

# -------------------------------------------------------------------
# STEP 1: TIME SORTING & TARGET CREATION
# -------------------------------------------------------------------
# Make sure event_time is a proper datetime object
final_df['event_time'] = pd.to_datetime(final_df['event_time'])

# Sort by session and then strictly by time (The "Movie" must be in order)
seq_df = final_df.sort_values(by=['user_session', 'event_time']).copy()

# Find which sessions actually resulted in a purchase (Our Target 'y')
# Hum ek dictionary bana rahe hain: {session_id: 1 ya 0}
session_targets = seq_df.groupby('user_session')['event_type'].apply(
    lambda events: 1 if 'purchase' in events.values else 0
).to_dict()


# -------------------------------------------------------------------
# STEP 2: PREVENTING "DATA LEAKAGE" (Cheating)
# -------------------------------------------------------------------
# WARNING: Hum GRU ke input mein 'purchase' event nahi bhej sakte. 
# Agar model ne input mein hi 'purchase' dekh liya, toh wo cheat karke output 1 de dega.
# Isliye hum sirf 'view' aur 'cart' events ko as a sequence use karenge.
seq_df = seq_df[seq_df['event_type'] != 'purchase'].copy()


# -------------------------------------------------------------------
# STEP 3: FEATURE ENGINEERING FOR SEQUENCE
# -------------------------------------------------------------------
# 1. Encoding Event Types (Neural Nets string nahi samajhte)
seq_df['is_view'] = (seq_df['event_type'] == 'view').astype(int)
seq_df['is_cart'] = (seq_df['event_type'] == 'cart').astype(int)

# 2. Extracting "Time Gap" (Hesitation Time between clicks)
seq_df['time_gap_seconds'] = seq_df.groupby('user_session')['event_time'].diff().dt.total_seconds().fillna(0)

# Time gaps bohot extreme ho sakte hain (e.g., 0 sec to 5000 sec). 
# Deep learning extreme numbers se confuse hota hai, isliye hum ispe Log transform lagayenge.
seq_df['log_time_gap'] = np.log1p(seq_df['time_gap_seconds'])

# 3. Scaling Price (0 to 1 ke beech lana zaroori hai weights stable rakhne ke liye)
scaler = MinMaxScaler()
seq_df['scaled_price'] = scaler.fit_transform(seq_df[['price']])


# -------------------------------------------------------------------
# STEP 4: BUILDING THE 3D ARRAY (The Magic Step)
# -------------------------------------------------------------------
# Jo 4 features hum GRU ko sikhana chahte hain:
sequence_features = ['is_view', 'is_cart', 'scaled_price', 'log_time_gap']

print("Grouping events into sequential lists...")
# Har session ka data ab ek 2D list ban jayega
session_groups = seq_df.groupby('user_session')[sequence_features].apply(lambda x: x.values.tolist())

# Aligning Targets (y) with the sequences
y = np.array([session_targets[session_id] for session_id in session_groups.index])

# -------------------------------------------------------------------
# STEP 5: PADDING TO FIXED LENGTH
# -------------------------------------------------------------------
# GRU matrix ki shape hamesha fix honi chahiye. Hum aakhri 15 actions par focus karenge.
MAX_STEPS = 15

print(f"Padding/Truncating sequences to {MAX_STEPS} steps...")
# padding='pre': Agar 15 se kam events hain, toh shuru mein [0,0,0,0] lag jayega.
# truncating='pre': Agar 50 events hain, toh purane 35 bhool jayega, sirf aakhri 15 yaad rakhega.
X = pad_sequences(session_groups.tolist(), maxlen=MAX_STEPS, padding='pre', truncating='pre', dtype='float32')

print("-" * 50)
print("✅ SUCCESS! Data is ready for GRU.")
print(f"🔹 Shape of Input (X): {X.shape}  --> (Total Sessions, Time Steps, Features)")
print(f"🔹 Shape of Target (y): {y.shape}    --> (Total Sessions,)")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

print("🧠 Building the GRU Neural Network...")

# STEP 1: Train-Test Split (Sequence data ko split karna)
# X shape: (Total_Sessions, 15, 4)
# y shape: (Total_Sessions,)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# STEP 2: Model Architecture Define Karna
model = Sequential()

# Layer 1: The Masking Layer (Sabse Zaroori!)
# Humne padding mein jo '0' dale the, ye layer GRU ko bolegi ki unhe ignore karo.
model.add(Masking(mask_value=0.0, input_shape=(X.shape[1], X.shape[2]))) 
# input_shape = (15 steps, 4 features)

# Layer 2: The GRU Brain
# 64 units ka matlab hai GRU ke paas 64 "neurons" ki memory capacity hai
model.add(GRU(units=64, return_sequences=False, activation='tanh'))

# Layer 3: Dropout (Overfitting rokne ke liye)
# Ye 20% neurons ko random time par band kar dega taaki model "ratta" na maare (memorize na kare)
model.add(Dropout(0.2))

# Layer 4: Dense (Processing Layer)
model.add(Dense(units=32, activation='relu'))

# Layer 5: Output Layer (The Final Decision)
# Sigmoid activation hamesha 0 se 1 ke beech ka probability score deta hai
model.add(Dense(units=1, activation='sigmoid'))


# STEP 3: Model Compile Karna
print("⚙️ Compiling the model...")
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', # Kyunki answer sirf Yes (1) ya No (0) hai
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')] # AUC recall/precision balance ke liye best hai
)

model.summary()

# STEP 4: Training (Fitting the model)
# EarlyStopping: Agar model seekhna band kar de, toh training khud rok do taaki time bache
early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True)

print("\n🚀 Starting Training...")
history = model.fit(
    X_train, y_train,
    epochs=15,          # Maximum 15 baar poora data dekhega
    batch_size=256,     # Ek baar mein 256 sessions process karega (speed ke liye)
    validation_split=0.2, 
    callbacks=[early_stop]
)

# STEP 5: Final Testing
print("\n📊 Evaluating on Test Data...")
test_loss, test_acc, test_auc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC Score: {test_auc:.4f}")